# Balanço do conteúdo salino

Este notebook verifica a conservação do esquema Euler implícito--Upwind usado no artigo. A grandeza $A\int_0^L C(x,t)\,dx$ é denominada **conteúdo salino integrado**, pois $C$ está em PSU; ela não representa diretamente massa de sal em quilogramas. Faça upload do arquivo `salt_intrusion_1d_v1.0.0.zip` antes de executar as células no Colab.

In [ ]:
!pip install ./salt_intrusion_1d_v1.0.0.zip --quiet --force-reinstall

Para os nós internos, o esquema possui a forma conservativa

$$
\frac{C_i^{n+1}-C_i^n}{\Delta t}+\frac{\widehat J_{i+1/2}^{n+1}-\widehat J_{i-1/2}^{n+1}}{\Delta x}=0,
$$

com

$$
\widehat J_{i+1/2}=v^+C_i+v^-C_{i+1}-D\frac{C_{i+1}-C_i}{\Delta x}.
$$

Logo, para $S_{\mathrm{int}}^n=A\Delta x\sum_{i=1}^{N-1}C_i^n$, o resíduo discreto deve ser próximo da precisão de máquina. Separadamente, avalia-se o balanço físico aproximado usando a regra dos trapézios e fluxos de fronteira com derivadas unilaterais.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from salt_intrusion_1d import simulate
from salt_intrusion_1d.article_update import article_config

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def globally_normalized_residual(residual, storage_rate, area, left_flux, right_flux):
    characteristic = np.abs(storage_rate) + area * (np.abs(left_flux) + np.abs(right_flux))
    scale = max(np.max(characteristic), np.finfo(float).eps)
    return np.abs(residual) / scale

def balance_diagnostics(result):
    cfg = result.config
    dt = cfg.dt_s
    area = cfg.cross_section_area_m2
    d_internal = np.diff(result.internal_salt_content_psu_m3) / dt
    d_trapezoidal = np.diff(result.trapezoidal_salt_content_psu_m3) / dt
    r_discrete = globally_normalized_residual(
        result.discrete_balance_residual_psu_m3_s[1:], d_internal, area,
        result.left_numerical_flux_psu_m_s[1:], result.right_numerical_flux_psu_m_s[1:]
    )
    r_physical = globally_normalized_residual(
        result.physical_balance_residual_psu_m3_s[1:], d_trapezoidal, area,
        result.left_physical_flux_psu_m_s[1:], result.right_physical_flux_psu_m_s[1:]
    )
    net_physical = area * (result.left_physical_flux_psu_m_s[1:] - result.right_physical_flux_psu_m_s[1:])
    accumulated_flux = np.concatenate(([0.0], np.cumsum(dt * net_physical)))
    storage_change = result.trapezoidal_salt_content_psu_m3 - result.trapezoidal_salt_content_psu_m3[0]
    accumulated_error = storage_change - accumulated_flux
    scale_final = max(np.max(np.abs(storage_change)), np.max(np.abs(accumulated_flux)), np.finfo(float).eps)
    summary = {
        "Q (m3/s)": area * cfg.river_velocity_m_s,
        "dx (m)": cfg.dx_m,
        "dt (s)": dt,
        "max r_discreto": np.max(r_discrete),
        "max r_fisico global": np.max(r_physical),
        "erro acumulado relativo final": abs(accumulated_error[-1]) / scale_final,
    }
    return summary, r_discrete, r_physical, storage_change, accumulated_flux, accumulated_error

## Cenários do artigo

As duas simulações usam os parâmetros finais do artigo: $L=50$ km, $\Delta x=12{,}5$ m, $\Delta t=7{,}5$ s e 60 ciclos de maré.

In [ ]:
results = {}
diagnostics = {}
for discharge in (10.0, 2.0):
    print(f"Executando Q={discharge:g} m3/s ...")
    results[discharge] = simulate(article_config(discharge, store_every_steps=30))
    diagnostics[discharge] = balance_diagnostics(results[discharge])

summary = pd.DataFrame([diagnostics[q][0] for q in (10.0, 2.0)])
display(summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for q, color in ((10.0, "tab:blue"), (2.0, "tab:orange")):
    result = results[q]
    _, r_disc, r_phys, storage, accumulated, error = diagnostics[q]
    time_cycles = result.times_s / result.config.tidal_period_s
    axes[0, 0].plot(time_cycles, result.trapezoidal_salt_content_psu_m3, color=color, label=f"Q={q:g}")
    axes[0, 1].semilogy(time_cycles[1:], np.maximum(r_disc, 1e-18), color=color, label=f"Q={q:g}")
    axes[1, 0].plot(time_cycles, storage, color=color, label=f"armazenamento, Q={q:g}")
    axes[1, 0].plot(time_cycles, accumulated, color=color, ls="--", label=f"fluxo acumulado, Q={q:g}")
    axes[1, 1].plot(time_cycles, error, color=color, label=f"Q={q:g}")
axes[0, 0].set(title="Conteúdo salino integrado", xlabel="ciclos de maré", ylabel="PSU m³")
axes[0, 1].set(title="Resíduo discreto normalizado", xlabel="ciclos de maré", ylabel="resíduo relativo")
axes[1, 0].set(title="Balanço físico acumulado", xlabel="ciclos de maré", ylabel="PSU m³")
axes[1, 1].set(title="Erro acumulado do balanço físico", xlabel="ciclos de maré", ylabel="PSU m³")
for ax in axes.flat:
    ax.legend(fontsize=8)
plt.show()

## Verificação por refinamento

A célula abaixo compara duas discretizações do cenário crítico. O resíduo discreto permanece no nível do arredondamento; o diagnóstico físico, que usa quadratura trapezoidal e derivadas unilaterais nas fronteiras, deve diminuir com o refinamento.

In [ ]:
refinement_rows = []
coarse = simulate(article_config(2.0, dx_m=25.0, dt_s=15.0, store_every_steps=60))
refinement_rows.append(balance_diagnostics(coarse)[0])
refinement_rows.append(diagnostics[2.0][0])
display(pd.DataFrame(refinement_rows))